In [3]:
import jieba
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import nltk
from nltk.corpus import stopwords

# 載入停用詞
首先，載入NLTK內建的中文停用詞為初始資料

In [4]:
nltk.download('stopwords')
stopWord = set(stopwords.words('chinese'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\styeh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 僅為示範綜合效果，繼續疊加中文停用詞表、哈爾濱工業大學停用詞表、百度停用詞表、四川大學機器智能實驗室停用詞表

In [ ]:
with open('./data/hit_stopwords.txt','r',encoding='utf-8') as f:   #載入哈爾濱工業大學停用詞
    for word in f.readlines():
        stopWord.update(word.strip())
with open('./data/cn_stopwords.txt','r',encoding='utf-8') as f:   #載入簡體中文停用詞
    for word in f.readlines():
        stopWord.update(word.strip())
with open('./data/baidu_stopwords.txt','r',encoding='utf-8') as f:   #載入百度停用詞
    for word in f.readlines():
        stopWord.update(word.strip())
with open('./data/scu_stopwords.txt','r',encoding='utf-8') as f:   #載入四川大學機器智能實驗室停用詞
    for word in f.readlines():
        stopWord.update(word.strip())

# 載入電影評論與其標籤，進行斷詞、去除停用詞

In [30]:
import re
def cutword(line):
    review=re.sub(r'[a-zA-Z0-9]*','',line)
    wordList=jieba.lcut(review,cut_all=False)
    return ' '.join([word for word in wordList if word not in stopWord and len(word)>1])
df_zh = pd.read_csv('./data/movie_data_zh.csv', encoding='utf-8')
df_zh['word_list']=df_zh['review'].apply(cutword)
X = df_zh['word_list'].to_numpy()
y = df_zh['sentiment'].to_numpy()
df_zh.head(10)

,review,sentiment,word_list
0,1974年，萬聖節惡作劇之夜，青少年瑪莎·莫克斯利瑪吉·格蕾絲搬到了康涅狄格州格林威治貝勒港...,1,萬聖節 惡作劇 之夜 青少年 瑪莎 克斯 利瑪吉 格蕾絲 康涅狄格州 格林威治 貝勒港 高級...
1,好吧，我真的很喜歡克里斯克里斯托佛森，他在電影時代慣用的輕鬆台詞幫助他實現了輕聲細語的低能量...,0,真的 喜歡 克里斯 克里斯托 佛森 電影 時代慣 能量 風格 他會 費力 一個 場景 消失 ...
2,劇透，如果你想看那部電影，請不要讀這篇文章，雖然這會浪費時間，但情節是如此可預測，無論你讀與...,0,劇透 那部 電影 篇文章 雖然 這會 浪費 時間 情節 預測 無論 區別 郊狼 醜陋 值得 ...
3,嗨，所有看過這部精彩電影的人，我相信你會像我一樣喜歡它，我喜歡這些歌曲，一旦你看過這個節目，...,1,看過 這部 精彩 電影 一樣 喜歡 喜歡 這些 歌曲 看過 這個 節目 唱歌 好像 是節 目...
4,我最近買了DVD，忘記了我是多麼討厭電影版的合唱台詞導演阿滕伯勒對故事所做的每一次改變都失敗...,0,忘記 多麼 討厭電 影版 合唱 台詞 導演 滕伯勒 故事 改變 失敗 因為 導演 卡西 的關...
5,讓 Braik 上演一場精彩的表演 最後，他和 Zorak 在 spac Ghost 之外過...,1,上演 一場 精彩 表演 之外 過著 生活 不得不 喜歡 這兩部 節目 它們 成人 游泳 目的...
6,內森·底特律·弗蘭克·西納特拉是紐約歷史最悠久的浮動擲骰子遊戲的經理，他需要 1000 美元...,1,內森 底特律 弗蘭克 西納 特拉 紐約 歷史 悠久 浮動 骰子 遊戲 經理 美元 來確 保一...
7,要在正確的背景下理解速成課程，您必須了解 80 年代的電視節目，大多數電視節目沒有任何意義，...,1,正確 背景 理解 速成 課程 必須 年代 電視 節目 大多 數電視 節目 沒有 意義 情境 ...
8,一段時間以來，我對查維斯反對全球化的立場印象深刻，但直到我在阿姆斯特丹國際紀錄片電影節上看到...,1,一段 時間 以來 查維斯反 全球化 立場 印象 深刻 阿姆斯特丹 國際紀 錄片 電影節 這部...
9,這部電影由雷尼哈林執導，芬蘭奇蹟史泰龍是加布沃克，貓和老鼠在山上與無情的恐怖分子雷尼哈林知道...,1,這部 電影 雷尼哈林 執導 芬蘭奇 史泰龍 加布 沃克 貓和 老鼠 山上 無情 恐怖分子 雷...


# 給予可能的各參數如下，經過交叉驗證尋找超參數最佳組合
**TfidfVectorizer的選項：vect__超參數名稱**

**LogisticRegression的選項：clf__超參數名稱**

      ngram_range=(1,2) 加入雙詞特徵，
      max_features 限制維度
      max_df=0.8:去除出現在80%文檔中的詞
      min_df=2:只保留至少出現在2個文本中的詞
    總共需要擬合驗證數: (fits)     
      超參數組合數: 2x2x2x2x2x3x2x2 + 1x1x1x2x2x2x2x2x2=384 + 64 = 448
      擬合5個folds: 448x5 = 2240
      超參數組合衝突: 2x2x2x2x2x1x2x1 = 64x5 = 320
      實際擬合驗證數: 2240 - 320 = 1920
X part 已完成斷詞，因此超參數`tokenizer`仍舊維持None。

**須注意運行時間的取捨**

      運行下列程式碼可能超過200分鐘，你可以考慮縮小資料規模(包括訓練樣本或甚至超參數`max_features`)節省時間，但可能得出效果較弱的模型，或者，減少個別超參數的元素來降低超參數的組合數，例如下列`param_grid`先測其中一個，去除不必要的參數元素後，再測另外一個。

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

tfidf = TfidfVectorizer(token_pattern=r'(?u)\b\w\w+\b',
                        max_df=0.8,min_df=2)
# Pipeline of tfidf+logisticregression
tfidf_lreg_pipeline = Pipeline([      
    ('vect', tfidf), 
    ('clf', LogisticRegression(random_state=0))])
param_grid = [{'vect__ngram_range': [(1,1),(1,2)],
               'vect__norm':['l1','l2'],
               'vect__max_features':[1000,1500],
               'vect__max_df':[0.8,0.9],
               'vect__min_df':[2,4],
               'clf__l1_ratio': [0,1,0.5],
               'clf__C': [1.0, 10.0],
               'clf__solver': ['liblinear','saga']},
              {'vect__ngram_range': [(1,1)],
               'vect__use_idf':[False],
               'vect__norm':[None],
               'vect__max_features':[1000,1500],
               'vect__max_df':[0.7,0.8],
               'vect__min_df':[2,6],
               'clf__l1_ratio': [0,1],
               'clf__C': [1.0, 10.0],
               'clf__solver': ['liblinear','lbfgs']},
              ]
gs_lr_tfidf = GridSearchCV(tfidf_lreg_pipeline, param_grid,     # 評估
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X, y)

上面程式分成5疊(5 fold)樣本進行交叉驗證正確率(accuracy)，其最後得出最佳準確度(`best_score_`)係為此5次的平均。

In [35]:
print('最佳超參數集: %s ' % gs_lr_tfidf.best_params_)
print('交叉驗證最佳準確度: %.3f' % gs_lr_tfidf.best_score_)

最佳超參數集: {'clf__C': 1.0, 'clf__l1_ratio': 0.5, 'clf__solver': 'saga', 'vect__max_df': 0.8, 'vect__max_features': 1500, 'vect__min_df': 2, 'vect__ngram_range': (1, 1), 'vect__norm': 'l2'} 
交叉驗證最佳準確度: 0.859


# 擬合、轉換
#### 使用如上交叉驗證得出的最佳超參數組：{'clf__C': 1.0, 'clf__l1_ratio': 0.5, 'clf__solver': 'saga', 'vect__max_df': 0.8, 'vect__max_features': 1500, 'vect__min_df': 2, 'vect__ngram_range': (1, 1), 'vect__norm': 'l2'} 

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01, random_state=42)
vectorizer = TfidfVectorizer(ngram_range=(1,1), max_features=1500,
                             norm='l2',
                             max_df=0.8,min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
model = LogisticRegression(C=1.0, l1_ratio=0.5,solver='saga')  # C 為正則化強度
model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.5
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [64]:
print("數字向量詞彙集:",vectorizer.vocabulary_.items())
y_pred_train = model.predict(X_train_tfidf)
print("訓練集準確率:",accuracy_score(y_train, y_pred_train))
X_test_tfidf = vectorizer.transform(X_test)
y_pred_test = model.predict(X_test_tfidf)
print("測試集準確率:",accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))
y_pred_proba=model.predict_proba(X_test_tfidf)
df_1 = pd.DataFrame({'真實標籤':y_test,'預測標籤':y_pred_test,'類別0機率':y_pred_proba[:,0],'類別1機率':y_pred_proba[:,1]})
df_1[(df_1['預測標籤']!=df_1['真實標籤'])]

數字向量詞彙集: dict_items([('史蒂夫', np.int64(302)), ('一位', np.int64(3)), ('紐約', np.int64(1095)), ('公寓', np.int64(207)), ('母親', np.int64(859)), ('一個', np.int64(4)), ('漂亮', np.int64(911)), ('名叫', np.int64(316)), ('結婚', np.int64(1101)), ('沒有', np.int64(879)), ('主角', np.int64(91)), ('那裡', np.int64(1392)), ('提供', np.int64(715)), ('已經', np.int64(504)), ('成功', np.int64(643)), ('冒險', np.int64(215)), ('東西', np.int64(819)), ('支持', np.int64(733)), ('這部', np.int64(1355)), ('電影會', np.int64(1439)), ('道德', np.int64(1371)), ('演出', np.int64(913)), ('這位', np.int64(1333)), ('觀眾', np.int64(1230)), ('看來', np.int64(1030)), ('這是', np.int64(1344)), ('一部', np.int64(32)), ('糟糕', np.int64(1090)), ('劇本', np.int64(240)), ('情節', np.int64(606)), ('台詞', np.int64(300)), ('雖然', np.int64(1432)), ('討人', np.int64(1237)), ('喜歡', np.int64(338)), ('很多', np.int64(560)), ('令人', np.int64(123)), ('討厭', np.int64(1238)), ('材料', np.int64(818)), ('角色', np.int64(1233)), ('時間', np.int64(781)), ('畢竟', np.int64(993)), ('起來', np.int64(1289)), (

,真實標籤,預測標籤,類別0機率,類別1機率
1,0,1,0.416703,0.583297
4,0,1,0.403269,0.596731
20,1,0,0.827140,0.172860
24,0,1,0.110171,0.889829
32,0,1,0.454087,0.545913
36,1,0,0.547536,0.452464
37,1,0,0.781841,0.218159
38,0,1,0.282799,0.717201
40,1,0,0.566315,0.433685
47,0,1,0.338434,0.661566


#### 繼續細分saga下features數與elastic net 係數觀其是否能有更好的效果

In [36]:
param_grid = [
              {'clf__l1_ratio': [0.3,0.5,0.8],
               'clf__C': [1.0],
               'clf__solver': ['saga'],
               'vect__max_df':[0.8],
               'vect__max_features':[1500,5000,10000,20000],
               'vect__min_df':[2],
               'vect__ngram_range': [(1,1)],
               'vect__norm':['l1','l2',None]}           
              ]
gs_lr_tfidf = GridSearchCV(tfidf_lreg_pipeline, param_grid,     # 評估
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X, y)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


d:\App\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf__C': [1.0], 'clf__l1_ratio': [0.3, 0.5, ...], 'clf__solver': ['saga'], 'vect__max_df': [0.8], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate 

In [38]:
print('最佳超參數集: %s ' % gs_lr_tfidf.best_params_)
print('交叉驗證最佳準確度: %.3f' % gs_lr_tfidf.best_score_)

最佳超參數集: {'clf__C': 1.0, 'clf__l1_ratio': 0.3, 'clf__solver': 'saga', 'vect__max_df': 0.8, 'vect__max_features': 20000, 'vect__min_df': 2, 'vect__ngram_range': (1, 1), 'vect__norm': None} 
交叉驗證最佳準確度: 0.874


#### 最佳超參數集如上'clf__l1_ratio': 0.3,'vect__max_features': 20000可獲得更好的學習效果
#### max_features=20000是上述交叉驗證給予[1500,5000,10000,20000]的最大值，若繼續加大max_features亦有可能提升準確度，但學習資源的使用亦會更昂貴，兩者需要尋求平衡

In [50]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X, y, test_size=0.01, random_state=42)
vectorizer_2 = TfidfVectorizer(ngram_range=(1,1), max_features=20000,
                             norm=None,
                             max_df=0.8,min_df=2)
X_train_tfidf_2 = vectorizer_2.fit_transform(X_train_2)
model_2 = LogisticRegression(C=1.0, l1_ratio=0.3,solver='saga')  # C 為正則化強度
model_2.fit(X_train_tfidf_2, y_train_2)

d:\App\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.3
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [61]:
print("數字向量詞彙集:",vectorizer_2.vocabulary_.items())
y_pred_train_2 = model_2.predict(X_train_tfidf_2)
print("訓練集準確率:",accuracy_score(y_train_2, y_pred_train_2))
X_test_tfidf_2 = vectorizer_2.transform(X_test_2)
y_pred_test_2 = model_2.predict(X_test_tfidf_2)
print("測試集準確率:",accuracy_score(y_test_2, y_pred_test_2))
print(classification_report(y_test_2, y_pred_test_2))
y_pred_proba_2=model_2.predict_proba(X_test_tfidf_2)
df_2 = pd.DataFrame({'真實標籤':y_test_2,'預測標籤':y_pred_test_2,'類別0機率':y_pred_proba_2[:,0],'類別1機率':y_pred_proba_2[:,1]})

數字向量詞彙集: dict_items([('史蒂夫', np.int64(4003)), ('布西', np.int64(6966)), ('一位', np.int64(13)), ('負的電', np.int64(16894)), ('紐約', np.int64(14442)), ('一間', np.int64(236)), ('破舊', np.int64(13726)), ('公寓', np.int64(2675)), ('母親', np.int64(11120)), ('幫忙', np.int64(7105)), ('支付', np.int64(9258)), ('房租', np.int64(8437)), ('一個', np.int64(14)), ('漂亮', np.int64(11962)), ('鄰居', np.int64(18209)), ('名叫', np.int64(4129)), ('安吉', np.int64(6067)), ('珍妮弗', np.int64(12818)), ('比爾斯', np.int64(11190)), ('結婚', np.int64(14525)), ('沒有', np.int64(11383)), ('擔任', np.int64(9203)), ('主角', np.int64(1081)), ('意外', np.int64(7989)), ('陌生人', np.int64(18747)), ('那裡', np.int64(18160)), ('提供', np.int64(9045)), ('資金', np.int64(16979)), ('已經', np.int64(6825)), ('成功', np.int64(8225)), ('冒險', np.int64(2772)), ('東西', np.int64(10516)), ('保時捷', np.int64(2028)), ('財務', np.int64(16902)), ('支持', np.int64(9259)), ('這部', np.int64(17744)), ('電影會', np.int64(19065)), ('阿道夫', np.int64(18728)), ('道德', np.int64(17942)), ('支柱', np.int64(9263)

In [ ]:
df_2[(df_2['預測標籤']!=df_2['真實標籤'])]

,真實標籤,預測標籤,類別0機率,類別1機率
1,0,1,0.150536,0.849464
4,0,1,0.340014,0.659986
20,1,0,0.937388,0.062612
24,0,1,0.261318,0.738682
28,1,0,0.537476,0.462524
37,1,0,0.590007,0.409993
38,0,1,0.137048,0.862952
40,1,0,0.668598,0.331402
47,0,1,0.451932,0.548068
48,1,0,0.630255,0.369745


In [68]:
param_grid = [
              {'clf__l1_ratio': [0.3,0.5,0.8],
               'clf__C': [1.0],
               'clf__solver': ['saga'],
               'vect__max_df':[0.8],
               'vect__max_features':[20000,30000],
               'vect__min_df':[2],
               'vect__ngram_range': [(1,1)],
               'vect__norm':['l1','l2',None]},
               {'clf__l1_ratio': [0],
               'clf__C': [1.0],
               'clf__solver': ['sag'],
               'vect__max_df':[0.8],
               'vect__max_features':[10000,20000,30000],
               'vect__min_df':[2],
               'vect__ngram_range': [(1,1)],
               'vect__norm':['l1','l2',None]}         
              ]
gs_lr_tfidf = GridSearchCV(tfidf_lreg_pipeline, param_grid,     # 評估
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X, y)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


d:\App\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf__C': [1.0], 'clf__l1_ratio': [0.3, 0.5, ...], 'clf__solver': ['saga'], 'vect__max_df': [0.8], ...}, {'clf__C': [1.0], 'clf__l1_ratio': [0], 'clf__solver': ['sag'], 'vect__max_df': [0.8], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the

加大max_features至30000提升準確度並不顯著，但學習資源的使用更昂貴，兩者需要尋求平衡

In [70]:
print('最佳超參數集: %s ' % gs_lr_tfidf.best_params_)
print('交叉驗證最佳準確度: %.3f' % gs_lr_tfidf.best_score_)

最佳超參數集: {'clf__C': 1.0, 'clf__l1_ratio': 0.3, 'clf__solver': 'saga', 'vect__max_df': 0.8, 'vect__max_features': 30000, 'vect__min_df': 2, 'vect__ngram_range': (1, 1), 'vect__norm': None} 
交叉驗證最佳準確度: 0.875


In [71]:
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X, y, test_size=0.01, random_state=42)
vectorizer_3 = TfidfVectorizer(ngram_range=(1,1), max_features=30000,
                             norm=None,
                             max_df=0.8,min_df=2)
X_train_tfidf_3 = vectorizer_3.fit_transform(X_train_3)
model_3 = LogisticRegression(C=1.0, l1_ratio=0.3,solver='saga')  # C 為正則化強度
model_3.fit(X_train_tfidf_3, y_train_3)

d:\App\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.3
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [72]:
y_pred_train_3 = model_3.predict(X_train_tfidf_3)
print("訓練集準確率:",accuracy_score(y_train_3, y_pred_train_3))
X_test_tfidf_3 = vectorizer_3.transform(X_test_3)
y_pred_test_3 = model_3.predict(X_test_tfidf_3)
print("測試集準確率:",accuracy_score(y_test_3, y_pred_test_3))

訓練集準確率: 0.9325359049708207
測試集準確率: 0.8834355828220859
